# Session 4: Statistical Analysis

**Introduction to Data Analysis with Python**

---

### What this session covers

| Part | Topic | Time |
|------|-------|------|
| A | Descriptive statistics and distributions | 10 min |
| B | Hypothesis tests: t-test and chi-square | 20 min |
| C | OLS regression with statsmodels | 20 min |
| D | Exercises | 10 min |

### Learning outcomes

By the end of this session you will be able to:

- Compute descriptive statistics and confidence intervals using `scipy.stats`
- Run an independent-samples t-test and interpret the output
- Run a chi-square test of association and calculate an effect size
- Fit an OLS regression model using `statsmodels` and read the summary table
- Map the `statsmodels` output onto the equivalent Stata `regress` output
- Export a regression results table to a file

### Dataset used in this session

`hse_adults.csv` — a cross-sectional survey of 600 English adults, modelled on the Health Survey for England.

| Column | Description |
|--------|-------------|
| `age` | Age in years |
| `sex` | Male / Female |
| `bmi` | Body mass index (some missing) |
| `physical_activity_days` | Days per week with at least 30 min of activity |
| `self_rated_health` | 1 = Excellent … 5 = Poor |
| `smoker` | Never / Ex-smoker / Current |
| `alcohol_units_weekly` | Units of alcohol per week (some missing) |
| `income_quintile` | 1 = lowest … 5 = highest |
| `education` | Highest qualification |
| `region` | English region |
| `systolic_bp` | Systolic blood pressure (mmHg) — main outcome |

> **Note:** `systolic_bp` is the outcome we will model. Normal is below 120; elevated is 120–129; high is 130 or above.

---

## Part A: Descriptive Statistics and Distributions

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

df = pd.read_csv("hse_adults.csv")
print(f"Loaded {len(df)} rows, {df.shape[1]} columns.")
df.head()

In [ ]:
df.describe()

### A.1 Confidence interval for a mean

The sample mean is our best estimate of the population mean, but it has uncertainty.
A 95% confidence interval (CI) tells us the range within which we are 95% confident the true population mean lies.

In [ ]:
bp = df["systolic_bp"].dropna()

mean_bp = bp.mean()
sem_bp  = stats.sem(bp)           # standard error of the mean
ci      = stats.t.interval(       # t-distribution CI (correct for finite samples)
    confidence=0.95,
    df=len(bp) - 1,
    loc=mean_bp,
    scale=sem_bp
)

print(f"Systolic blood pressure:")
print(f"  n    = {len(bp)}")
print(f"  Mean = {mean_bp:.1f} mmHg")
print(f"  95% CI: ({ci[0]:.1f}, {ci[1]:.1f})")

### A.2 Testing for normality

Many statistical tests assume the variable is approximately normally distributed.
The Shapiro-Wilk test checks this formally.

In [ ]:
stat, p = stats.shapiro(bp)
print(f"Shapiro-Wilk test for normality:")
print(f"  W = {stat:.4f},  p = {p:.4f}")
print()
if p > 0.05:
    print("p > 0.05: no strong evidence against normality.")
else:
    print("p <= 0.05: evidence of non-normality — consider transforming or using non-parametric tests.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(bp, bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Systolic Blood Pressure")
axes[0].set_xlabel("Systolic BP (mmHg)")

stats.probplot(bp, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot (normal distribution)")

plt.tight_layout()
plt.show()

> **Reading the Q-Q plot:** if the points follow the diagonal line closely, the distribution is approximately normal. Deviations at the ends indicate heavy tails or skew.

---

## Part B: Hypothesis Tests

### B.1 Independent-samples t-test

A t-test asks: is the mean of a continuous variable significantly different between two groups?

**Research question:** Is mean systolic blood pressure significantly different between men and women?

```stata
* Stata
ttest systolic_bp, by(sex)
```

In [ ]:
# Split into two groups
male_bp   = df.loc[df["sex"] == "Male",   "systolic_bp"].dropna()
female_bp = df.loc[df["sex"] == "Female", "systolic_bp"].dropna()

print(f"Males:   n={len(male_bp)},  mean={male_bp.mean():.1f},  sd={male_bp.std():.1f}")
print(f"Females: n={len(female_bp)}, mean={female_bp.mean():.1f},  sd={female_bp.std():.1f}")

In [ ]:
# Run the t-test
# equal_var=False uses Welch's t-test, which doesn't assume equal variances
t_stat, p_val = stats.ttest_ind(male_bp, female_bp, equal_var=False)

print(f"Welch's independent-samples t-test:")
print(f"  t  = {t_stat:.3f}")
print(f"  p  = {p_val:.4f}")
print()
if p_val < 0.05:
    print("p < 0.05: statistically significant difference between groups.")
else:
    print("p >= 0.05: no statistically significant difference at the 5% level.")

In [ ]:
# Visualise the difference
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="sex", y="systolic_bp", ax=ax, order=["Male", "Female"])
ax.set_title("Systolic Blood Pressure by Sex")
ax.set_xlabel("")
ax.set_ylabel("Systolic BP (mmHg)")
plt.tight_layout()
plt.show()

> **Stata comparison:**
>
> | Output | Stata | Python |
> |--------|-------|--------|
> | Test statistic | `t` | `t_stat` |
> | p-value (two-tailed) | `Pr(|T| > |t|)` | `p_val` |
> | Group means | shown in output table | `male_bp.mean()` etc. |
> | Degrees of freedom | shown automatically | `len(male_bp) + len(female_bp) - 2` |

### B.2 Chi-square test of association

A chi-square test asks: is there an association between two categorical variables?

**Research question:** Is smoking status associated with sex?

```stata
* Stata
tabulate sex smoker, chi2
```

In [ ]:
# Build a contingency table — like Stata's tabulate with two variables
contingency = pd.crosstab(df["sex"], df["smoker"])
print("Contingency table:")
print(contingency)

In [ ]:
# Run the chi-square test
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)

print(f"Chi-square test of association:")
print(f"  chi2 = {chi2:.3f}")
print(f"  df   = {dof}")
print(f"  p    = {p_chi2:.4f}")
print()

# Cramer's V — effect size for chi-square (0 = no association, 1 = perfect)
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
print(f"  Cramer's V (effect size) = {cramers_v:.3f}")
print("  (0.1 = small, 0.3 = medium, 0.5 = large)")

In [ ]:
# Show row percentages to see the pattern more clearly
row_pct = pd.crosstab(df["sex"], df["smoker"], normalize="index").round(3) * 100
print("Row percentages (%):")
print(row_pct.round(1))

---

## Part C: OLS Regression with statsmodels

OLS (Ordinary Least Squares) regression models the relationship between a continuous outcome and one or more predictors.

**Research question:** What predicts systolic blood pressure?

```stata
* Stata
regress systolic_bp age i.sex bmi physical_activity_days i.smoker
```

In Python we use `statsmodels`, which produces output very similar to Stata's `regress`.

In [ ]:
import statsmodels.formula.api as smf

# Patsy formula syntax — almost identical to Stata
# C(variable) tells statsmodels it is categorical (creates dummies automatically)
formula = "systolic_bp ~ age + C(sex) + bmi + physical_activity_days + C(smoker)"

# Fit the model — dropna handles missing bmi automatically
model  = smf.ols(formula=formula, data=df).fit()

# Print the full results table
print(model.summary())

### Reading the output — Stata comparison

| Stata column | statsmodels column | What it means |
|-------------|-------------------|---------------|
| `Coef.` | `coef` | The estimated effect of a one-unit increase in the predictor |
| `Std. Err.` | `std err` | Uncertainty in the coefficient estimate |
| `t` | `t` | Test statistic (coef / std err) |
| `P>|t|` | `P>|t|` | p-value — is the effect statistically significant? |
| `[95% Conf. Interval]` | `[0.025  0.975]` | 95% confidence interval for the coefficient |
| `R-squared` | `R-squared` | Proportion of outcome variance explained by the model |

### Interpreting the coefficients

- **age**: each additional year of age is associated with an increase of approximately 0.4 mmHg in systolic BP
- **C(sex)[T.Male]**: men have higher BP on average than women (the reference category)
- **bmi**: each unit increase in BMI is associated with higher BP
- **physical_activity_days**: more active days per week is associated with lower BP
- **C(smoker)[T.Current]**: current smokers have higher BP than never-smokers (the reference)

In [ ]:
# Pull out just the key numbers cleanly
results_table = pd.DataFrame({
    "Coefficient": model.params,
    "Std Error":   model.bse,
    "p-value":     model.pvalues,
    "CI lower":    model.conf_int()[0],
    "CI upper":    model.conf_int()[1],
}).round(3)

results_table

In [ ]:
# Model fit statistics
print(f"R-squared:          {model.rsquared:.3f}")
print(f"Adj. R-squared:     {model.rsquared_adj:.3f}")
print(f"F-statistic:        {model.fvalue:.2f}  (p = {model.f_pvalue:.4f})")
print(f"Number of obs:      {int(model.nobs)}")

### C.1 Visualising the fit

A residual plot checks whether the model assumptions are met. The residuals (observed minus predicted) should be randomly scattered around zero.

In [ ]:
fitted    = model.fittedvalues
residuals = model.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs fitted
axes[0].scatter(fitted, residuals, alpha=0.4, s=20)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals vs Fitted Values")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")

# Distribution of residuals
sns.histplot(residuals, bins=30, kde=True, ax=axes[1])
axes[1].set_title("Distribution of Residuals")
axes[1].set_xlabel("Residual (mmHg)")

plt.tight_layout()
plt.show()

### C.2 Exporting the results table

For a report or dissertation, export the results to a file you can paste into Word or LaTeX.

In [ ]:
# Export as CSV — easy to paste into Excel or Word
results_table.to_csv("regression_results.csv")
print("Saved: regression_results.csv")

# Export as plain text (like Stata's outreg)
with open("regression_results.txt", "w") as f:
    f.write(model.summary().as_text())
print("Saved: regression_results.txt")

---

## Quick Reference: Stata to Python

| Task | Stata | Python |
|------|-------|--------|
| t-test (two groups) | `ttest y, by(group)` | `stats.ttest_ind(a, b, equal_var=False)` |
| Chi-square test | `tabulate a b, chi2` | `stats.chi2_contingency(pd.crosstab(a, b))` |
| Contingency table | `tabulate a b` | `pd.crosstab(df["a"], df["b"])` |
| OLS regression | `regress y x1 x2` | `smf.ols("y ~ x1 + x2", data=df).fit()` |
| Categorical predictor | `i.varname` | `C(varname)` in formula |
| Regression summary | (printed automatically) | `model.summary()` |
| R-squared | shown in output | `model.rsquared` |
| Coefficients | `e(b)` | `model.params` |
| Confidence intervals | `regress, level(95)` | `model.conf_int()` |
| Export results | `outreg2` | `results_table.to_csv()` |
| Fitted values | `predict yhat` | `model.fittedvalues` |
| Residuals | `predict resid, resid` | `model.resid` |

---

## Part D: Exercises

---

### Exercise 1 — Confidence interval

Compute a 95% confidence interval for mean BMI. How does it compare to the WHO overweight threshold of 25?

*(Hint: use the same `stats.t.interval()` approach as Part A, but on `df["bmi"].dropna()`.)*

In [ ]:
# Exercise 1 — your code here

---

### Exercise 2 — t-test

Test whether mean systolic blood pressure differs significantly between current smokers and never-smokers.

1. Extract the two groups
2. Run a Welch t-test
3. Report the means, t-statistic, p-value, and a one-sentence conclusion

In [ ]:
# Exercise 2 — your code here

---

### Exercise 3 — Chi-square test

Test whether self-rated health (`self_rated_health`, treated as a categorical variable with values 1–5) is associated with smoking status.

1. Create the contingency table
2. Run the chi-square test
3. Calculate Cramer's V
4. Write two sentences interpreting the result

In [ ]:
# Exercise 3 — your code here

---

### Exercise 4 — Regression (stretch goal)

Fit a new OLS model predicting `systolic_bp`, but this time add `income_quintile` as a predictor (treat it as continuous, not categorical).

1. Write the formula and fit the model
2. Is `income_quintile` a statistically significant predictor?
3. What is the R-squared compared to the original model?
4. Produce a residual plot for the new model

In [ ]:
# Exercise 4 — your code here

---

## Wrap-up

### What you covered across all four sessions

**Session 1** — Python and Jupyter: the environment, basic Python, loading and inspecting data with pandas

**Session 2** — Data wrangling: cleaning, recoding, grouped summaries, merging files

**Session 3** — Exploratory data analysis and visualisation: distributions, relationships, correlation heatmaps, saving figures, CRAFT

**Session 4** — Statistical analysis: descriptive stats and confidence intervals, t-tests, chi-square tests, OLS regression, exporting results

### Suggested next steps

- **Logistic regression** — for binary outcomes (e.g. smoker: yes/no): `smf.logit()` in statsmodels
- **Survey weights** — the real HSE dataset uses complex survey weights; `statsmodels` supports weighted regression
- **Interactive visualisation** — plotly produces charts you can zoom and hover over, ideal for exploratory work
- **Reproducible workflows** — organise your analysis as a series of scripts or notebooks that run from raw data to final output in one step
- **Version control** — commit your notebooks to Git so you have a full history of every change you made

### Further reading

- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) — free online
- [statsmodels documentation](https://www.statsmodels.org) — worked examples for regression and time series
- [The Programming Historian](https://programminghistorian.org) — tutorials for humanities and social science researchers

### Save your work

**File → Save Notebook** (or Cmd+S).